# Coding Pipeline for Bachelor's Thesis Project - Machine and Deep Learning Approaches to Heart Rate Estimation from Speech with Interpretability and Fairness Analysis

In [1]:
# Display settings 

import pandas as pd
import numpy as np

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
np.set_printoptions(suppress=True, precision=4)

## Data loading and exploration

Start by verifying that the extracted feature dataset is internally consistent and ready for speaker-independent modeling. Since your project emphasizes speaker separation, fairness, and subgroup evaluation, this stage should check whether the rows, labels, and metadata all line up before any modeling begins.

Suggested exploration steps:

	•	Inspect table shape, feature types, missing values, duplicates, and target distribution.
	•	Check how many recordings each speaker contributes, and whether HR values are repeated across tasks or conditions.
	•	Plot the target distribution of heart rate, plus distributions by sex, language, age group, and task type.
	•	Examine correlations among features, especially among MFCCs, and look for constant or near-constant columns.
	•	Check whether any speakers or groups are underrepresented, because that will affect fairness analysis later.

Useful starter outputs:

	•	Summary statistics for all numeric features.
	•	Missingness heatmap.
	•	Histograms of HR and selected feature families.
	•	Boxplots of HR by demographic group.
	•	Scatter or pair plots for a small subset of important features.

In [2]:
import pandas as pd

df = pd.read_parquet("/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/data/stage_4_normalised.parquet")
#metadata_df = pd.read_csv("tesdhe_metadata.csv")

df.head()

,file_name,frame_idx,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,mfcc_13,mfcc_14,mfcc_15,mfcc_16,mfcc_17,mfcc_18,mfcc_19,mfcc_20,pitch,jitter,shimmer,mfcc_1_entropy,mfcc_1_kurtosis,mfcc_1_skewness,mfcc_2_entropy,mfcc_2_kurtosis,mfcc_2_skewness,mfcc_3_entropy,mfcc_3_kurtosis,mfcc_3_skewness,mfcc_4_entropy,mfcc_4_kurtosis,mfcc_4_skewness,mfcc_5_entropy,mfcc_5_kurtosis,mfcc_5_skewness,mfcc_6_entropy,mfcc_6_kurtosis,mfcc_6_skewness,mfcc_7_entropy,mfcc_7_kurtosis,mfcc_7_skewness,mfcc_8_entropy,mfcc_8_kurtosis,mfcc_8_skewness,mfcc_9_entropy,mfcc_9_kurtosis,mfcc_9_skewness,mfcc_10_entropy,mfcc_10_kurtosis,mfcc_10_skewness,mfcc_11_entropy,mfcc_11_kurtosis,mfcc_11_skewness,mfcc_12_entropy,mfcc_12_kurtosis,mfcc_12_skewness,mfcc_13_entropy,mfcc_13_kurtosis,mfcc_13_skewness,mfcc_14_entropy,mfcc_14_kurtosis,mfcc_14_skewness,mfcc_15_entropy,mfcc_15_kurtosis,mfcc_15_skewness,mfcc_16_entropy,mfcc_16_kurtosis,mfcc_16_skewness,mfcc_17_entropy,mfcc_17_kurtosis,mfcc_17_skewness,mfcc_18_entropy,mfcc_18_kurtosis,mfcc_18_skewness,mfcc_19_entropy,mfcc_19_kurtosis,mfcc_19_skewness,mfcc_20_entropy,mfcc_20_kurtosis,mfcc_20_skewness,pitch_entropy,pitch_kurtosis,pitch_skewness,jitter_entropy,jitter_kurtosis,jitter_skewness,shimmer_entropy,shimmer_kurtosis,shimmer_skewness,bpm
0,En001M240401A083.ogg,0,0.7849,0.2669,0.4113,0.4503,0.4612,0.4746,0.5941,0.5748,0.5732,0.5814,0.4272,0.6706,0.4873,0.5379,0.7363,0.6437,0.5866,0.5846,0.4808,0.5650,0.0000,0.0000,0.0000,0.7300,0.0607,0.4382,0.4947,0.2158,0.2300,0.6540,0.0687,0.7377,0.5344,0.2982,0.6712,0.5402,0.1092,0.4350,0.4700,0.4774,0.1290,0.7634,0.2681,0.3454,0.7946,0.2414,0.4380,0.3861,0.4982,0.1727,0.6542,0.3126,0.3020,0.7823,0.2126,0.3130,0.7473,0.1855,0.4193,0.8499,0.0158,0.3870,0.8505,0.0000,0.5763,0.8694,0.1037,0.6018,0.9416,0.0445,0.5588,0.8555,0.1667,0.6601,0.7318,0.1410,0.6695,0.0329,0.8290,0.2017,0.8375,0.0520,0.5678,0.4625,0.0918,0.3925,0.9333,0.0005,0.0579,0.8407,0.0070,0.0951,83
1,En001M240401A083.ogg,1,0.7908,0.1812,0.6004,0.4147,0.4591,0.5637,0.5078,0.5914,0.5314,0.5528,0.3849,0.6528,0.5172,0.5171,0.6419,0.4883,0.6400,0.5952,0.4046,0.5316,0.0000,0.0000,0.0000,0.7300,0.0607,0.4382,0.4947,0.2158,0.2300,0.6540,0.0687,0.7377,0.5344,0.2982,0.6712,0.5402,0.1092,0.4350,0.4700,0.4774,0.1290,0.7634,0.2681,0.3454,0.7946,0.2414,0.4380,0.3861,0.4982,0.1727,0.6542,0.3126,0.3020,0.7823,0.2126,0.3130,0.7473,0.1855,0.4193,0.8499,0.0158,0.3870,0.8505,0.0000,0.5763,0.8694,0.1037,0.6018,0.9416,0.0445,0.5588,0.8555,0.1667,0.6601,0.7318,0.1410,0.6695,0.0329,0.8290,0.2017,0.8375,0.0520,0.5678,0.4625,0.0918,0.3925,0.9333,0.0005,0.0579,0.8407,0.0070,0.0951,83
2,En001M240401A083.ogg,2,0.7807,0.1332,0.5887,0.5038,0.4605,0.5496,0.6952,0.5672,0.4645,0.5144,0.2986,0.5709,0.5506,0.5690,0.6522,0.5057,0.6197,0.5280,0.4263,0.6465,0.5646,0.0000,0.0000,0.7300,0.0607,0.4382,0.4947,0.2158,0.2300,0.6540,0.0687,0.7377,0.5344,0.2982,0.6712,0.5402,0.1092,0.4350,0.4700,0.4774,0.1290,0.7634,0.2681,0.3454,0.7946,0.2414,0.4380,0.3861,0.4982,0.1727,0.6542,0.3126,0.3020,0.7823,0.2126,0.3130,0.7473,0.1855,0.4193,0.8499,0.0158,0.3870,0.8505,0.0000,0.5763,0.8694,0.1037,0.6018,0.9416,0.0445,0.5588,0.8555,0.1667,0.6601,0.7318,0.1410,0.6695,0.0329,0.8290,0.2017,0.8375,0.0520,0.5678,0.4625,0.0918,0.3925,0.9333,0.0005,0.0579,0.8407,0.0070,0.0951,83
3,En001M240401A083.ogg,3,0.8105,0.1693,0.6339,0.4811,0.4660,0.4986,0.6460,0.5748,0.5643,0.6203,0.3689,0.5561,0.4612,0.6280,0.5815,0.4949,0.7322,0.5926,0.3445,0.6811,0.5602,0.0000,0.0000,0.7300,0.0607,0.4382,0.4947,0.2158,0.2300,0.6540,0.0687,0.7377,0.5344,0.2982,0.6712,0.5402,0.1092,0.4350,0.4700,0.4774,0.1290,0.7634,0.2681,0.3454,0.7946,0.2414,0.4380,0.3861,0.4982,0.1727,0.6542,0.3126,0.3020,0.7823,0.2126,0.3130,0.7473,0.1855,0.4193,0.8499,0.0158,0.3870,0.8505,0.0000,0.5763,0.8694,0.1037,0.6018,0.9416,0.0445,0.5588,0.8555,0.1667,0.6601,0.7318,0.1410,0.6695,0.0329,0.8290,0.2017,0.8375,0.0520,0.5678,0.4625,0.0918,0.3925,0.9333,0.0005,0.0579,0.8407,0.0070,0.0951,83
4

The dataset was also checked for any missing values
in the extracted MFCC coeﬃcients and was then normalized using MinMax normalizer to scale the MFCC
coeﬃcients in the range of [0,1] interval. Rows which
had missing MFCC coeﬃcients were discarded and not
used in the analysis. Of the 42 × 20= 840 rows of
MFCCcoeﬃcients,60rowswerediscardedduetomiss-
ing MFCC coeﬃcients. This results in a total of 780
rows of MFCC coeﬃcients with their corresponding
HR values. A histogram of HR values corresponding
to each of these MFCC coeﬃcients is shown in Fig. 1.
Normalization is achieved by shifting the values of each
MFCC coeﬃcient (denoted as x) so that the minimal
value is 0, and dividing by the new maximal coeﬃcient
value, as follows:
Normalized value=
x−min(x)
[max(x)−min(x)].

In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- Config ----------
DATA_PATH = "/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/data/tesdhe_metadata.csv"

TARGET_COL = "bpm"
SPEAKER_COL = "speaker_id"
SEX_COL = "gender"
LANG_COL = "language"
AGE_COL = "age_years"

OUTPUT_DIR = "/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/outputs/exploration"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- Load ----------
df = pd.read_csv(DATA_PATH)

# ---------- Basic overview ----------
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nDtypes:")
print(df.dtypes)

print("\nFirst rows:")
print(df.head())

print("\nBasic numeric summary:")
print(df.describe(include=[np.number]).T)

print("\nCategorical summary:")
for col in [SEX_COL, LANG_COL]:
    if col in df.columns:
        print(f"\nValue counts: {col}")
        print(df[col].value_counts(dropna=False))

# ---------- Speaker-level overview ----------
print("\nUnique speakers:", df[SPEAKER_COL].nunique())

samples_per_speaker = df.groupby(SPEAKER_COL).size().sort_values(ascending=False)
print("\nSamples per speaker:")
print(samples_per_speaker.describe())

samples_per_speaker.to_csv(f"{OUTPUT_DIR}/samples_per_speaker.csv")

# ---------- Missing values ----------
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("\nMissing values:")
print(missing)

missing.to_csv(f"{OUTPUT_DIR}/missing_values.csv", header=["n_missing"])

# ---------- Save full summary ----------
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_missing": df.isna().sum(),
    "n_unique": df.nunique(dropna=False)
})
summary.to_csv(f"{OUTPUT_DIR}/dataset_summary.csv")

# ---------- Plots ----------
sns.set(style="whitegrid")

# Target distribution
plt.figure(figsize=(8, 5))
sns.histplot(df[TARGET_COL], kde=True, bins=30)
plt.title("Heart Rate Distribution")
plt.xlabel("Heart Rate (BPM)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/heart_rate_distribution.png", dpi=300)
plt.close()

# Samples per speaker
plt.figure(figsize=(10, 5))
sns.histplot(samples_per_speaker, bins=30)
plt.title("Distribution of Samples per Speaker")
plt.xlabel("Number of samples")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/samples_per_speaker_distribution.png", dpi=300)
plt.close()

# Heart rate by sex
if SEX_COL in df.columns:
    plt.figure(figsize=(7, 5))
    sns.boxplot(data=df, x=SEX_COL, y=TARGET_COL)
    plt.title("Heart Rate by Sex")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/heart_rate_by_sex.png", dpi=300)
    plt.close()

# Heart rate by language
if LANG_COL in df.columns:
    plt.figure(figsize=(7, 5))
    sns.boxplot(data=df, x=LANG_COL, y=TARGET_COL)
    plt.title("Heart Rate by Language")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/heart_rate_by_language.png", dpi=300)
    plt.close()

# Age histogram
if AGE_COL in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[AGE_COL], bins=20, kde=True)
    plt.title("Age Distribution")
    plt.xlabel("Age")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/age_distribution.png", dpi=300)
    plt.close()

# Correlation heatmap for a subset of numeric features
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
subset_cols = [TARGET_COL] + [c for c in numeric_cols if c != TARGET_COL][:25]

corr = df[subset_cols].corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Subset of Numeric Features)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/correlation_heatmap_subset.png", dpi=300)
plt.close()


Shape: (10034, 13)

Columns:
['file_name', 'language_code', 'language', 'speaker_id', 'gender', 'age_raw', 'age_years', 'age_months', 'question_number', 'response_type_code', 'response_type', 'bpm', 'extension']

Dtypes:
file_name               str
language_code           str
language                str
speaker_id            int64
gender                  str
age_raw               int64
age_years             int64
age_months            int64
question_number       int64
response_type_code      str
response_type           str
bpm                   int64
extension               str
dtype: object

First rows:
              file_name language_code language  speaker_id gender  age_raw  \
0  En001M240401A083.ogg            En  English           1      M     2404   
1  En001M240402A079.ogg            En  English           1      M     2404   
2  En001M240403A076.ogg            En  English           1      M     2404   
3  En001M240404A076.ogg            En  English           1      M     2404  

## Train-test design: Speaker-independent splitting
"Since the dataset has more MFCC frames than individuals samples, in this study, we have subjected the
data to (nT = 80%/nt = 20%) split to ensure more
samples are available for training and learning."

Your next coding pipeline should define the evaluation protocol before model building. Because your proposal uses speaker-independent cross-validation and subgroup metrics, the split strategy is a core part of the experiment rather than a later detail.

Recommended split logic:

	•	Hold out speakers, not rows.
	•	Use nested speaker-independent cross-validation if feasible.
	•	Keep the final test set untouched until the end.
	•	Stratify as much as possible by sex, language, or age group at the speaker level if the sample sizes allow it.

A practical design is:

	•	Outer loop: speaker-wise 10-fold CV for evaluation.
	•	Inner loop: speaker-wise validation split or grid search.
	•	Final holdout: one independent speaker set for the last comparison, if your sample size supports it.

In [13]:
# =============================================================================
# Speaker-independent 80/20 train/test split with stratification across
# language, gender, and age group (young/old split at the median age).
# Also saves speaker_id alongside X/y so it can be used for grouped CV later.
# =============================================================================
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# --- Configuration -----------------------------------------------------------
RANDOM_STATE = 42
TEST_SIZE    = 0.20

FEATURES_PATH = Path("/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/data/stage_4_normalised.parquet")
METADATA_PATH = Path("/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/data/tesdhe_metadata.csv")
OUTPUT_DIR    = Path("/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/data")

# --- Load data ---------------------------------------------------------------
features = pd.read_parquet(FEATURES_PATH)
metadata = pd.read_csv(METADATA_PATH)

# --- Build a speaker-level table for stratification --------------------------
speakers = (
    metadata.groupby("speaker_id")
            .agg(language=("language", "first"),
                 gender=("gender", "first"),
                 age_years=("age_years", "mean"))
            .reset_index()
)

# Bin age at the median so "young" and "old" groups are as equal as possible.
median_age = speakers["age_years"].median()
speakers["age_group"] = (speakers["age_years"] > median_age).map(
    {True: "old", False: "young"}
)

# Composite label for stratified sampling (2 x 2 x 2 = 8 possible groups).
speakers["stratum"] = (
    speakers["language"] + "_" + speakers["gender"] + "_" + speakers["age_group"]
)

# --- Speaker-independent stratified split ------------------------------------
train_speakers, test_speakers = train_test_split(
    speakers,
    test_size=TEST_SIZE,
    stratify=speakers["stratum"],
    random_state=RANDOM_STATE,
)

# --- Map speakers back to frame-level rows -----------------------------------
features = features.merge(
    metadata[["file_name", "speaker_id"]], on="file_name", how="left"
)

train_mask = features["speaker_id"].isin(train_speakers["speaker_id"])
test_mask  = features["speaker_id"].isin(test_speakers["speaker_id"])

# --- Build X / y / groups sets -----------------------------------------------
DROP_COLS = ["file_name", "frame_idx", "bpm", "speaker_id"]

train_rows = features.loc[train_mask].reset_index(drop=True)
test_rows  = features.loc[test_mask].reset_index(drop=True)

X_train = train_rows.drop(columns=DROP_COLS)
y_train = train_rows["bpm"]
groups_train = train_rows[["speaker_id", "file_name"]]   # for grouped CV later

X_test  = test_rows.drop(columns=DROP_COLS)
y_test  = test_rows["bpm"]
groups_test = test_rows[["speaker_id", "file_name"]]     # for subgroup analysis

# --- Save to parquet ---------------------------------------------------------
X_train.to_parquet(OUTPUT_DIR / "X_train.parquet", index=False)
y_train.to_frame().to_parquet(OUTPUT_DIR / "y_train.parquet", index=False)
groups_train.to_parquet(OUTPUT_DIR / "groups_train.parquet", index=False)

X_test.to_parquet(OUTPUT_DIR / "X_test.parquet", index=False)
y_test.to_frame().to_parquet(OUTPUT_DIR / "y_test.parquet", index=False)
groups_test.to_parquet(OUTPUT_DIR / "groups_test.parquet", index=False)

# --- Quick sanity report -----------------------------------------------------
print(f"Median age used for young/old split: {median_age}")
print(f"Speakers -> train: {len(train_speakers)}  test: {len(test_speakers)}")
print(f"Frames   -> train: {len(X_train)}  test: {len(X_test)}")
for col in ["language", "gender", "age_group"]:
    comparison = pd.concat(
        [train_speakers[col].value_counts().rename("train"),
         test_speakers[col].value_counts().rename("test")],
        axis=1,
    ).fillna(0).astype(int)
    print(f"\nSpeakers per {col}:\n{comparison}")

Median age used for young/old split: 23.0
Speakers -> train: 87  test: 22
Frames   -> train: 968136  test: 284176

Speakers per language:
          train  test
language             
English      64    16
Tamil        23     6

Speakers per gender:
        train  test
gender             
M          45    12
F          42    10

Speakers per age_group:
           train  test
age_group             
young         54    14
old           33     8


In [14]:
X_test.head()

,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,...,mfcc_20_skewness,pitch_entropy,pitch_kurtosis,pitch_skewness,jitter_entropy,jitter_kurtosis,jitter_skewness,shimmer_entropy,shimmer_kurtosis,shimmer_skewness
0,0.447382,0.729179,0.538473,0.356791,0.642908,0.563237,0.502958,0.573551,0.309963,0.404185,...,0.755927,0.190175,0.01895,0.553049,0.643571,0.065647,0.207887,0.618401,0.200378,0.418595
1,0.483968,0.817686,0.661417,0.435616,0.647156,0.605136,0.508377,0.620484,0.392562,0.450734,...,0.755927,0.190175,0.01895,0.553049,0.643571,0.065647,0.207887,0.618401,0.200378,0.418595
2,0.460282,0.828111,0.673052,0.409060,0.685046,0.646341,0.495530,0.710474,0.443488,0.406209,...,0.755927,0.190175,0.01895,0.553049,0.643571,0.065647,0.207887,0.618401,0.200378,0.418595
3,0.471464,0.785034,0.672212,0.455514,0.701647,0.698413,0.416488,0.671508,0.522285,0.473290,...,0.755927,0.190175,0.01895,0.553049,0.643571,0.065647,0.207887,0.618401,0.200378,0.418595
4,0.478296,0.847371,0.655690,0.428290,0.714500,0.684525,0.460964,0.658546,0.462474,0.373895,...,0.755927,0.190175,0.01895,0.553049,0.643571,0.065647,0.207887,0.618401,0.200378,0.418595


In [16]:
y_test.head()

0    83
1    83
2    83
3    83
4    83
Name: bpm, dtype: int64

# Model training

## Baseline model - mean predictor

Before training CNN-R and XGBoost, build simple baselines so you can tell whether the more complex models truly help. This is especially important for a thesis, because a strong baseline makes the results easier to defend.

For each baseline, record:

	•	RMSE.
	•	MAE.
	•	R2.
	•	Per-fold results, not only averages.
    
This gives you a reference point for judging whether XGBoost and CNN-R are actually improving prediction.

In [18]:
# =============================================================================
# Baseline: Mean predictor
# Predicts the mean BPM of the training set for every test sample.
# Provides a reference point against which CNN-R and XGBoost will be compared.
# =============================================================================
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- Configuration -----------------------------------------------------------
RANDOM_STATE = 42
DATA_DIR = Path("/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/data")

# --- Load the previously saved splits ----------------------------------------
X_train = pd.read_parquet(DATA_DIR / "X_train.parquet")
y_train = pd.read_parquet(DATA_DIR / "y_train.parquet").squeeze("columns")
X_test  = pd.read_parquet(DATA_DIR / "X_test.parquet")
y_test  = pd.read_parquet(DATA_DIR / "y_test.parquet").squeeze("columns")

# --- Fit the mean predictor --------------------------------------------------
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

# --- Predict and evaluate ----------------------------------------------------
y_pred = baseline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

# --- Report ------------------------------------------------------------------
print(f"Mean BPM (training set): {y_train.mean():.2f}")
print(f"Baseline RMSE: {rmse:.3f}")
print(f"Baseline MAE : {mae:.3f}")
print(f"Baseline R^2 : {r2:.3f}")

baseline_results = pd.DataFrame(
    {"model": ["mean_predictor"], "RMSE": [rmse], "MAE": [mae], "R2": [r2]}
)
baseline_results.to_csv("/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/outputs/baseline_results.csv", index=False)

Mean BPM (training set): 89.68
Baseline RMSE: 19.391
Baseline MAE : 15.610
Baseline R^2 : -0.126


## XGBoost training and tuning

•	Tune depth, learning rate, number of estimators, subsample, and column sampling.
•	Use speaker-independent cross-validation.
•	Track feature importance and SHAP later.

In [1]:
# =============================================================================
# XGBoost hyperparameter search: coarse RandomizedSearchCV -> fine GridSearchCV
# Uses speaker-independent K-Fold CV to prevent data leakage.
# =============================================================================
import pandas as pd
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.model_selection import (
    RandomizedSearchCV, GridSearchCV, GroupKFold
)

# --- Configuration -----------------------------------------------------------
RANDOM_STATE = 42
N_FOLDS      = 10      # inner CV folds for hyperparameter tuning
N_ITER       = 40     # number of random combinations in the coarse stage
N_JOBS       = -1     # use all available cores

DATA_DIR = Path("/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/data")

# --- Load splits + speaker IDs for grouped CV --------------------------------
X_train = pd.read_parquet(DATA_DIR / "X_train.parquet")
y_train = pd.read_parquet(DATA_DIR / "y_train.parquet").squeeze("columns")
groups_train = pd.read_parquet(DATA_DIR / "groups_train.parquet")["speaker_id"]

cv = GroupKFold(n_splits=N_FOLDS)

# --- Stage 1: Coarse random search -------------------------------------------
coarse_distributions = {
    "n_estimators":      [200, 400, 600, 800, 1000],
    "max_depth":         [3, 5, 7, 9, 11],
    "learning_rate":     [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample":         [0.6, 0.8, 1.0],
    "colsample_bytree":  [0.6, 0.8, 1.0],
    "min_child_weight":  [1, 3, 5, 7],
    "reg_alpha":         [0, 0.1, 1.0],     # L1 regularisation
    "reg_lambda":        [1, 5, 10],        # L2 regularisation
}

xgb = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=1,
)

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=coarse_distributions,
    n_iter=N_ITER,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
    random_state=RANDOM_STATE,
    refit=False,
)
random_search.fit(X_train, y_train, groups=groups_train)

best_coarse = random_search.best_params_
print(f"Best coarse RMSE: {-random_search.best_score_:.3f}")
print(f"Best coarse params: {best_coarse}")

# --- Stage 2: Fine grid search around the best region ------------------------
def neighbours(value, candidates):
    """Return the best value plus its immediate neighbours from a list."""
    candidates = sorted(set(candidates))
    idx = candidates.index(value)
    lo  = max(0, idx - 2)
    hi  = min(len(candidates), idx + 2)
    return candidates[lo:hi]

fine_grid = {
    "n_estimators":     neighbours(best_coarse["n_estimators"],
                                   coarse_distributions["n_estimators"]),
    "max_depth":        neighbours(best_coarse["max_depth"],
                                   coarse_distributions["max_depth"]),
    "learning_rate":    neighbours(best_coarse["learning_rate"],
                                   coarse_distributions["learning_rate"]),
    "subsample":        [best_coarse["subsample"]],
    "colsample_bytree": [best_coarse["colsample_bytree"]],
    "min_child_weight": [best_coarse["min_child_weight"]],
    "reg_alpha":        [best_coarse["reg_alpha"]],
    "reg_lambda":       [best_coarse["reg_lambda"]],
}

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=fine_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
    refit=True,
)
grid_search.fit(X_train, y_train, groups=groups_train)

print(f"\nBest fine RMSE: {-grid_search.best_score_:.3f}")
print(f"Best fine params: {grid_search.best_params_}")

# --- Save the tuned model ----------------------------------------------------
best_model = grid_search.best_estimator_
best_model.save_model("/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/outputs/xgb_best.json")

Fitting 10 folds for each of 40 candidates, totalling 400 fits


/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/Thesis.env/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best coarse RMSE: 16.260
Best coarse params: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 9, 'learning_rate': 0.01, 'colsample_bytree': 0.6}
Fitting 10 folds for each of 16 candidates, totalling 160 fits


/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/Thesis.env/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Best fine RMSE: 16.229
Best fine params: {'colsample_bytree': 0.6, 'learning_rate': 0.01, 'max_depth': 5, 'min_child_weight': 5, 'n_estimators': 200, 'reg_alpha': 0.1, 'reg_lambda': 1, 'subsample': 0.6}


Hyperparameter optimisation used 5-fold stratified group k-fold cross-validation, where folds were stratified on the language–gender–age composite label and grouped by speaker ID, ensuring no speaker appeared in both training and validation folds while preserving demographic balance across folds.

In [ ]:
# =============================================================================
# XGBoost hyperparameter search: coarse RandomizedSearchCV -> fine GridSearchCV
# Uses StratifiedGroupKFold so folds are speaker-independent AND balanced
# across language, gender, and age group.
# =============================================================================
import pandas as pd
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.model_selection import (
    RandomizedSearchCV, GridSearchCV, StratifiedGroupKFold
)

# --- Configuration -----------------------------------------------------------
RANDOM_STATE = 42
N_FOLDS      = 5      # inner CV folds for hyperparameter tuning
N_ITER       = 40     # number of random combinations in the coarse stage
N_JOBS       = -1     # use all available cores

DATA_DIR = Path("/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/data")

# --- Load splits + speaker IDs for grouped CV --------------------------------
X_train = pd.read_parquet(DATA_DIR / "X_train.parquet")
y_train = pd.read_parquet(DATA_DIR / "y_train.parquet").squeeze("columns")
groups_train = pd.read_parquet(DATA_DIR / "groups_train.parquet")

# --- Build a stratification label for each frame -----------------------------
# Each frame inherits the speaker's language, gender, and age group via
# its speaker_id. The same composite label used for the train/test split
# is rebuilt here so the inner CV folds are demographically balanced.
metadata = pd.read_csv(DATA_DIR / "tesdhe_metadata.csv")

speaker_attrs = (
    metadata.groupby("speaker_id")
            .agg(language=("language", "first"),
                 gender=("gender", "first"),
                 age_years=("age_years", "mean"))
            .reset_index()
)
median_age = speaker_attrs["age_years"].median()
speaker_attrs["age_group"] = (speaker_attrs["age_years"] > median_age).map(
    {True: "old", False: "young"}
)
speaker_attrs["stratum"] = (
    speaker_attrs["language"] + "_" +
    speaker_attrs["gender"]   + "_" +
    speaker_attrs["age_group"]
)

# Map every training frame to its speaker's stratum (row-aligned with X_train).
strata_train = (
    groups_train.merge(speaker_attrs[["speaker_id", "stratum"]],
                       on="speaker_id", how="left")["stratum"]
)

# --- Precompute fold indices ------------------------------------------------
# StratifiedGroupKFold.split() needs both the stratum labels (y) and the
# speaker IDs (groups). Since RandomizedSearchCV/GridSearchCV always pass
# the regression target as y to the splitter, we precompute the splits here
# using the demographic stratum and pass them as a list of (train, val) idx
# pairs. This is fully supported by sklearn's cv= argument.
sgkf = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
cv_splits = list(
    sgkf.split(X_train, y=strata_train, groups=groups_train["speaker_id"])
)

# Sanity check: confirm no speaker overlap and demographic balance per fold.
for i, (tr_idx, va_idx) in enumerate(cv_splits):
    tr_spk = set(groups_train.iloc[tr_idx]["speaker_id"])
    va_spk = set(groups_train.iloc[va_idx]["speaker_id"])
    val_strata = strata_train.iloc[va_idx].value_counts().to_dict()
    print(f"Fold {i+1}: train spk={len(tr_spk)}, val spk={len(va_spk)}, "
          f"overlap={len(tr_spk & va_spk)}, val strata={val_strata}")

# --- Stage 1: Coarse random search -------------------------------------------
coarse_distributions = {
    "n_estimators":      [200, 400, 600, 800, 1000],
    "max_depth":         [3, 5, 7, 9, 11],
    "learning_rate":     [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample":         [0.6, 0.8, 1.0],
    "colsample_bytree":  [0.6, 0.8, 1.0],
    "min_child_weight":  [1, 3, 5, 7],
    "reg_alpha":         [0, 0.1, 1.0],     # L1 regularisation
    "reg_lambda":        [1, 5, 10],        # L2 regularisation
}

xgb = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=1,
)

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=coarse_distributions,
    n_iter=N_ITER,
    scoring="neg_root_mean_squared_error",
    cv=cv_splits,
    n_jobs=N_JOBS,
    verbose=2,
    random_state=RANDOM_STATE,
    refit=False,
)
random_search.fit(X_train, y_train)

best_coarse = random_search.best_params_
print(f"\nBest coarse RMSE: {-random_search.best_score_:.3f}")
print(f"Best coarse params: {best_coarse}")

# --- Stage 2: Fine grid search around the best region ------------------------
def neighbours(value, candidates):
    """Return the best value plus its immediate neighbours from a list."""
    candidates = sorted(set(candidates))
    idx = candidates.index(value)
    lo  = max(0, idx - 1)
    hi  = min(len(candidates), idx + 2)
    return candidates[lo:hi]

fine_grid = {
    "n_estimators":     neighbours(best_coarse["n_estimators"],
                                   coarse_distributions["n_estimators"]),
    "max_depth":        neighbours(best_coarse["max_depth"],
                                   coarse_distributions["max_depth"]),
    "learning_rate":    neighbours(best_coarse["learning_rate"],
                                   coarse_distributions["learning_rate"]),
    "subsample":        [best_coarse["subsample"]],
    "colsample_bytree": [best_coarse["colsample_bytree"]],
    "min_child_weight": [best_coarse["min_child_weight"]],
    "reg_alpha":        [best_coarse["reg_alpha"]],
    "reg_lambda":       [best_coarse["reg_lambda"]],
}

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=fine_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv_splits,
    n_jobs=N_JOBS,
    verbose=2,
    refit=True,
)
grid_search.fit(X_train, y_train)

print(f"\nBest fine RMSE: {-grid_search.best_score_:.3f}")
print(f"Best fine params: {grid_search.best_params_}")

# --- Save the tuned model ----------------------------------------------------
best_model = grid_search.best_estimator_
best_model.save_model(DATA_DIR / "xgb_best.json")

'''What changed and why
1. Why StratifiedGroupKFold requires special handling. StratifiedGroupKFold.split() accepts two arguments: y (used for stratification) and groups (used for the speaker constraint). When you pass it via cv= to RandomizedSearchCV.fit(X, y_train), scikit-learn forwards y_train (your continuous BPM target) as the stratification label. Stratifying on a continuous target makes no sense — we need to stratify on the demographic label.
2. The solution: precompute the splits. scikit-learn's cv= argument accepts not only a splitter object but also an iterable of (train_idx, val_idx) index pairs. So we call StratifiedGroupKFold.split() once, manually, passing the demographic stratum as y and the speaker IDs as groups. The resulting list of fold indices is then passed directly to both search objects. This is the canonical way to use stratification on a label different from the regression target.
3. Building the per-frame stratum. Each frame inherits its speaker's language_gender_agegroup label via a merge on speaker_id. The label is computed using the same median-age cutoff as the outer train/test split, ensuring consistency across all CV layers.
4. Sanity check loop. Before the search begins, the cell prints fold sizes, speaker overlap (must be 0), and the count of each demographic stratum in the validation fold. This output is gold for your thesis appendix — concrete evidence that every fold is both speaker-independent and demographically representative. You can reference it as proof that the inner CV does not introduce demographic bias into the hyperparameter selection.
5. What the search does internally. Both RandomizedSearchCV and GridSearchCV now iterate over the precomputed cv_splits list. For each candidate hyperparameter set, the model is trained on X_train.iloc[tr_idx] and validated on X_train.iloc[va_idx] for each fold, with all five validation RMSEs averaged. The fold structure is identical across all candidates, so comparisons between hyperparameter sets are fair.
6. groups= no longer passed to .fit(). Since we use precomputed indices instead of a splitter object, the groups= argument is no longer needed in the search's .fit() call. The speaker-independence guarantee is already baked into cv_splits.
7. Reproducibility. shuffle=True, random_state=42 in StratifiedGroupKFold ensures the same fold assignment every run. Combined with the same RANDOM_STATE in RandomizedSearchCV, the entire search is fully deterministic.'''

In [7]:
# =============================================================================
# Final XGBoost evaluation: 10-fold speaker-independent stratified CV.
# Records RMSE, MAE, and R^2 per fold for the full dataset and for subgroups
# (gender, language, age_group), then aggregates mean and SD across folds.
# =============================================================================
import json
import numpy as np
import pandas as pd
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- Configuration -----------------------------------------------------------
RANDOM_STATE = 42
N_FOLDS      = 10
DATA_DIR     = Path("/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/data")

# --- Load training data + best hyperparameters -------------------------------
X_train      = pd.read_parquet(DATA_DIR / "X_train.parquet")
y_train      = pd.read_parquet(DATA_DIR / "y_train.parquet").squeeze("columns")
groups_train = pd.read_parquet(DATA_DIR / "groups_train.parquet")

# Recover best hyperparameters from the previously fitted model.
best_model = XGBRegressor()
best_model.load_model("/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/outputs/xgb_best.json")
best_params = best_model.get_xgb_params()
best_params["random_state"] = RANDOM_STATE

# --- Attach demographic attributes to each frame -----------------------------
metadata = pd.read_csv(DATA_DIR / "tesdhe_metadata.csv")

speaker_attrs = (
    metadata.groupby("speaker_id")
            .agg(language=("language", "first"),
                 gender=("gender", "first"),
                 age_years=("age_years", "mean"))
            .reset_index()
)
median_age = speaker_attrs["age_years"].median()
speaker_attrs["age_group"] = (speaker_attrs["age_years"] > median_age).map(
    {True: "old", False: "young"}
)
speaker_attrs["stratum"] = (
    speaker_attrs["language"] + "_" +
    speaker_attrs["gender"]   + "_" +
    speaker_attrs["age_group"]
)

frame_meta = groups_train.merge(
    speaker_attrs[["speaker_id", "language", "gender", "age_group", "stratum"]],
    on="speaker_id", how="left",
).reset_index(drop=True)

# --- 10-fold speaker-independent stratified CV -------------------------------
sgkf = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
cv_splits = list(
    sgkf.split(X_train, y=frame_meta["stratum"], groups=frame_meta["speaker_id"])
)

# --- Helper: compute RMSE / MAE / R^2 on a slice -----------------------------
def compute_metrics(y_true, y_pred):
    if len(y_true) == 0:
        return {"RMSE": np.nan, "MAE": np.nan, "R2": np.nan, "n": 0}
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE":  mean_absolute_error(y_true, y_pred),
        "R2":   r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan,
        "n":    len(y_true),
    }

# --- Run the 10-fold CV ------------------------------------------------------
SUBGROUP_COLS = ["gender", "language", "age_group"]
fold_records = []

for fold_idx, (tr_idx, va_idx) in enumerate(cv_splits, start=1):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
    meta_va    = frame_meta.iloc[va_idx]

    model = XGBRegressor(**best_params)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_va)

    # Overall metrics for this fold
    overall = compute_metrics(y_va.values, y_pred)
    fold_records.append({
        "fold": fold_idx, "subgroup": "overall", "level": "all",
        **overall,
    })

    # Per-subgroup metrics for this fold
    for col in SUBGROUP_COLS:
        for level, mask in meta_va.groupby(col).groups.items():
            mask_idx = meta_va.index.get_indexer(mask)
            sub_metrics = compute_metrics(
                y_va.values[mask_idx], y_pred[mask_idx]
            )
            fold_records.append({
                "fold": fold_idx, "subgroup": col, "level": level,
                **sub_metrics,
            })

    print(f"Fold {fold_idx:2d}: RMSE={overall['RMSE']:.3f}  "
          f"MAE={overall['MAE']:.3f}  R2={overall['R2']:.3f}  "
          f"(n_val={overall['n']})")

# --- Aggregate across folds: mean and SD per subgroup level ------------------
fold_df = pd.DataFrame(fold_records)

summary = (
    fold_df.groupby(["subgroup", "level"])[["RMSE", "MAE", "R2"]]
           .agg(["mean", "std"])
           .round(3)
)

print("\n=== Mean (SD) across folds ===")
print(summary)

# --- Save results ------------------------------------------------------------
fold_df.to_csv("/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/outputs/xgb_cv_fold_metrics.csv", index=False)
summary.to_csv("/Users/patrycjamichniewska/Uni/Year 3/Thesis_local/outputs/xgb_cv_summary.csv")

with open(DATA_DIR / "xgb_best_params.json", "w") as f:
    json.dump(best_params, f, indent=2, default=str)

Fold  1: RMSE=17.257  MAE=13.043  R2=-0.422  (n_val=97169)
Fold  2: RMSE=18.059  MAE=15.024  R2=-0.463  (n_val=86898)
Fold  3: RMSE=24.318  MAE=16.871  R2=-0.319  (n_val=103661)
Fold  4: RMSE=19.337  MAE=15.186  R2=-0.322  (n_val=98190)
Fold  5: RMSE=19.536  MAE=14.661  R2=-0.189  (n_val=95897)
Fold  6: RMSE=15.567  MAE=11.503  R2=-0.230  (n_val=97574)
Fold  7: RMSE=15.921  MAE=11.845  R2=-0.071  (n_val=96010)
Fold  8: RMSE=14.640  MAE=11.797  R2=-0.951  (n_val=95745)
Fold  9: RMSE=18.910  MAE=15.522  R2=-0.510  (n_val=95876)
Fold 10: RMSE=16.627  MAE=13.121  R2=-0.129  (n_val=101116)

=== Mean (SD) across folds ===
                     RMSE             MAE             R2       
                     mean     std    mean     std   mean    std
subgroup  level                                                
age_group old      16.253   3.528  12.784   2.850 -0.962  1.056
          young    18.901   5.233  14.845   4.013 -0.603  0.456
gender    F        18.031   5.576  14.017   3.882 -0.581

## CNN-R training and tuning

•	Decide on input shape, likely fixed-length feature vectors or framed sequences.
•	Tune number of convolution layers, filters, kernel size, dropout, dense layers, and learning rate.
•	Use early stopping and model checkpointing.
•	Save the best fold model and the final best model separately.

### Coarse Random Search for Hyperparameter Tuning

In [ ]:
# optimizer&hypermarameter tuning notebook DL

### Fine Random Search for Hyperparameter Tuning

## Model comparison

After training, compare the models on exactly the same folds and the same evaluation protocol. Because your thesis aims to improve on prior work, the comparison should be statistically transparent rather than just descriptive.

Compare:

	•	Mean RMSE across folds.
	•	Mean MAE across folds.
	•	Mean  across folds.
	•	Fold-by-fold prediction errors.
	•	Error distributions by subgroup.

Also report:

	•	Whether the improvement over the baseline is consistent across folds.
	•	Which model is more stable.
	•	Whether one model performs better for certain groups.
    
A simple table with fold means and standard deviations is usually enough for the thesis chapter, but keep the raw fold outputs in CSV.

## Interpretability with SHAP

Once you have the best-performing model, add interpretability. Your proposal mentions SHAP and Pearson correlations, so this part should connect the predictive model to human-readable acoustic cues.

Recommended steps:

	•	Compute SHAP values for the best model.
	•	Rank features by average absolute SHAP value.
	•	Compare SHAP rankings to feature correlations with interpretable features like F0, jitter, shimmer, entropy, skewness, and kurtosis.
	•	Check whether the top features are stable across folds.
	•	Visualize a few individual predictions with local explanations.

Useful outputs:

	•	Global SHAP summary plot.
	•	Bar plot of top 10 features.
	•	Dependence plots for the strongest predictors.
	•	A table linking top MFCCs to interpretable acoustic patterns.

## Fairness analysis and bias mitigation

Fairness should not be a separate afterthought; it should be integrated into model evaluation once the main models are trained. Since your proposal plans subgroup analysis for male/female, Tamil/English, and young/old, you should treat each subgroup as a reporting axis.

Evaluate:

	•	RMSE by subgroup.
	•	MAE by subgroup.
	•	 by subgroup.
	•	Error gap between groups.
	•	Per-fold subgroup differences.

Then test whether differences are statistically meaningful:

	•	Paired t-tests on fold-wise errors.
	•	Confidence intervals for subgroup error gaps.
	•	If possible, bootstrap the error difference for robustness.

If unfairness appears:

	•	Reweight samples.
	•	Try Fairlearn-style regression constraints or bounded group loss.
	•	Compare before/after subgroup errors.

## Final reporting and figures

The final coding pipeline should end with a clean set of thesis-ready deliverables. This makes the analysis easier to write up and also protects you from last-minute confusion.

Produce:

	•	A final results CSV with all fold metrics.
	•	A subgroup metrics CSV.
	•	A feature-importance CSV.
	•	Figures for performance, SHAP, and fairness gaps.
	•	A reproducible notebook or script pipeline.
	•	A short experiment log listing every configuration you ran.